In [0]:
%run ./02_utility

# gold.fact_cash_transactions (refined star schema)

In [0]:
log_checkpoint("gold_fact_cash_transactions", "in_progress")

df_ct_silver = spark.table(f"{catalog}.{silver_schema}.cash_transactions") \
    .filter(col("_batch") <= batch_id)

df_fact_ct = df_ct_silver.select(
    col("CT_CA_ID").cast("bigint").alias("SK_AccountID"),
    col("CT_DTS").alias("TransactionDatetime"),
    col("CT_AMT").alias("Amount"),
    col("CT_NAME").alias("Description"),
    col("_batch"),
    current_timestamp().alias("_load_ts"),
    lit(run_id).alias("_run_id"),
)


In [0]:
target_table = f"{catalog}.{gold_schema}.fact_cash_transactions"

if spark.catalog.tableExists(target_table):
    delta_target = DeltaTable.forName(spark, target_table)
    delta_target.alias("tgt").merge(
        df_fact_ct.alias("src"),
        "tgt.SK_AccountID = src.SK_AccountID AND tgt.TransactionDatetime = src.TransactionDatetime AND tgt.Amount = src.Amount AND tgt.Description = src.Description"
    ) \
    .whenNotMatchedInsertAll().execute()

    print("merge for gold fact table done")

else:
    df_fact_ct.write \
        .saveAsTable(target_table)
    print("gold.fact_cash_transactions, table creation complete")

In [0]:
fact_ct_count = spark.table(target_table).count()
print(f"Total rows: {fact_ct_count}")

log_audit("gold_fact_cash_transactions", "merge", fact_ct_count)
log_checkpoint("gold_fact_cash_transactions", "completed", fact_ct_count)

# gold.dim_account : scd -2 from bronze

In [0]:
log_checkpoint("gold_dim_account", "in_progress")

### 1st source: batch 1
df_xml_events = spark.table(f"{catalog}.{bronze_schema}.customermgmt") \
        .filter(
            (col("ActionType").isin("NEW","ADDACCT", "UPDACCT", "CLOSEACCT"))
            & (col("CA_ID").isNotNull())
            & (regexp_extract(col("_batch"), r"(\d+)", 1).cast("INT") <= int(batch_id))
        ) \
        .select(
            col("CA_ID").cast("BIGINT").alias("AccountID"),
            col("C_ID").cast("BIGINT").alias("CA_C_ID"),
            col("CA_B_ID").cast("BIGINT").alias("CA_B_ID"),
            col("CA_NAME").alias("AccountDesc"),
            col("CA_TAX_ST").cast("TINYINT").alias("TaxStatus"),
            # Revert to ActionType mapping because CA_ST_ID does not exist in the XML
            when(col("ActionType").isin("NEW", "ADDACCT"), lit("ACTV"))
             .when(col("ActionType") == "CLOSEACCT", lit("CLOS"))
             .otherwise(lit(None))
             .alias("Status"),
            col("ActionTS").cast("TIMESTAMP").alias("event_ts"),
            col("_batch"),
        )

In [0]:
df_batchdate = spark.table(f"{catalog}.{silver_schema}.batchdate") \
        .select(col("batchid").cast("string").alias("_batch_num"), col("batchdate").cast("timestamp").alias("batch_timestamp"))

if batch_id in ("2", "3"):
    df_cdc_events = spark.table(f"{catalog}.{bronze_schema}.account") \
        .filter(
            (col("CDC_FLAG") != "D") & (regexp_extract(col("_batch"), r"(\d+)", 1).cast("INT") <= int(batch_id))
        )
else:
    df_cdc_events = None

In [0]:
if df_cdc_events is not None:
        df_cdc_events = df_cdc_events.withColumn("_batch_num", regexp_extract(col("_batch"), r"(\d+)", 1)) \
            .join(df_batchdate, "_batch_num", "left") \
            .select(
                col("CA_ID").cast("BIGINT").alias("AccountID"),
                col("CA_C_ID").cast("BIGINT").alias("CA_C_ID"),
                col("CA_B_ID").cast("BIGINT").alias("CA_B_ID"),
                col("CA_NAME").alias("AccountDesc"),
                col("CA_TAX_ST").cast("TINYINT").alias("TaxStatus"),
                col("CA_ST_ID").alias("Status"),
                col("batch_timestamp").alias("event_ts"),
                col("_batch"),
            )

In [0]:
if df_cdc_events is not None:
    df_all_events = df_xml_events.unionByName(df_cdc_events)
    print(f"Total account events: {df_all_events.count()}")
    print(f"XML events: {df_xml_events.count()}")
    print(f"CDC events: {df_cdc_events.count()}")
else:
    df_all_events = df_xml_events
    print(f"Batch 1: XML events only: {df_all_events.count()}")

In [0]:
# scd-2 window function
ff_window = Window.partitionBy("AccountID").orderBy("event_ts").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
df_filled = df_all_events \
    .select(
        "AccountID",
        last("CA_C_ID", ignorenulls=True).over(ff_window).alias("CA_C_ID"),
        last("CA_B_ID", ignorenulls=True).over(ff_window).alias("CA_B_ID"),
        last("AccountDesc", ignorenulls=True).over(ff_window).alias("AccountDesc"),
        last("TaxStatus", ignorenulls=True).over(ff_window).alias("TaxStatus"),
        last("Status", ignorenulls=True).over(ff_window).alias("Status"),
        "event_ts",
        "_batch",
    )

In [0]:
dedup_window = Window.partitionBy("AccountID", col("event_ts").cast("date")).orderBy(col("event_ts").desc())
    
df_filled_dedup = df_filled.withColumn("_rn", row_number().over(dedup_window)) \
    .filter(col("_rn") == 1) \
    .drop("_rn")
    
print(f"Count after forward-fill: {df_filled.count()}")
print(f"After same-day dedup: {df_filled_dedup.count():,} rows")


In [0]:
df_prep = df_filled_dedup.withColumn("EffectiveDate", col("event_ts").cast("date")) \
        .withColumn(
            "record_hash",
            md5(concat_ws("|", coalesce(col("AccountDesc"), lit("")), coalesce(col("CA_B_ID").cast("STRING"), lit("")), coalesce(col("Status"), lit("")) ))
        )

change_window = Window.partitionBy("AccountID").orderBy("EffectiveDate")

df_scd2_filtered = df_prep.withColumn("prev_hash", lag("record_hash").over(change_window)) \
    .filter(col("prev_hash").isNull() | (col("record_hash") != col("prev_hash"))) \
    .drop("prev_hash")

scd2_window = Window.partitionBy("AccountID").orderBy("EffectiveDate")

# OLD: df_scd2 = df_prep \
# Using df_scd2_filtered to exclude unchanged records and get correct count
df_scd2 = df_scd2_filtered \
        .withColumn("EndDate", coalesce(
            lead("EffectiveDate").over(scd2_window),
            lit("9999-12-31").cast("date")
        )) \
        .withColumn("version_number", row_number().over(scd2_window)) \
        .withColumn("IsCurrent", when(col("EndDate") == lit("9999-12-31").cast("date"), lit(True)).otherwise(lit(False))) \
        .filter(col("EffectiveDate") < col("EndDate")) 


In [0]:
print(f"SCD 2 version; rows -> {df_scd2.count()}")
print(f"Unique accounts {df_scd2.select('AccountID').distinct().count()}")

In [0]:
df_dim_customer = spark.table(f"{catalog}.{gold_schema}.dim_customer") \
        .select(
            col("customerid").alias("cust_c_id"),
            col("sk_customerid"),
            col("effectivedate").alias("cust_eff_date"),
            coalesce(col("enddate"), lit("9999-12-31").cast("DATE")).alias("cust_end_date"),
        )

In [0]:
print(f"dim_customer versions loaded: {df_dim_customer.count()}")
print(f"  with NULL EndDate (coalesced): {spark.table(f'{catalog}.{gold_schema}.dim_customer').filter(col('enddate').isNull()).count()}")

In [0]:
df_expanded = df_scd2.alias("a") \
    .join(
        df_dim_customer.alias("c"),
        (col("a.CA_C_ID") == col("c.cust_c_id")) & 
        (col("c.cust_eff_date") < col("a.EndDate")) & 
        (col("c.cust_end_date") > col("a.EffectiveDate")), 
        "inner"
    ) \
    .select(
        col("a.AccountID"),
        col("a.CA_C_ID"),
        col("a.CA_B_ID"),
        col("a.AccountDesc"),
        col("a.TaxStatus"),
        col("a.Status"),
        col("a._batch"),
        col("a.record_hash"),
        greatest(col("a.EffectiveDate"), col("c.cust_eff_date")).alias("EffectiveDate"),
        least(col("a.EndDate"), col("c.cust_end_date")).alias("EndDate"),
        col("c.sk_customerid").alias("SK_CustomerID"),
    )

In [0]:
df_null_customer = df_scd2.alias("a") \
        .join(
            df_dim_customer.alias("c"),
            (col("a.CA_C_ID") == col("c.cust_c_id")) & 
            (col("c.cust_eff_date") < col("a.EndDate")) & 
            (col("c.cust_end_date") > col("a.EffectiveDate")),
            "left_anti"
        ) \
        .select(
            "AccountID", "CA_C_ID", "CA_B_ID", "AccountDesc", "TaxStatus", "Status", 
            "_batch", "record_hash", "EffectiveDate", "EndDate", 
            lit(None).cast("bigint").alias("SK_CustomerID"),
        )


In [0]:
df_all_expanded = df_expanded.unionByName(df_null_customer)
expanded_count = df_expanded.count()
null_count = df_null_customer.count()

print(f"SCD2 count: {df_scd2.count()}")
print(f"Expanded customer count: {expanded_count:,}")
print(f"  NULL customer accounts: {null_count:,}")

In [0]:
scd2_expanded_window = Window.partitionBy("AccountID").orderBy("EffectiveDate")
    
df_final_scd2 = df_all_expanded \
    .withColumn("version_number", row_number().over(scd2_expanded_window)) \
    .withColumn("IsCurrent", when(col("EndDate") == lit("9999-12-31").cast("date"), lit(True)).otherwise(lit(False))) \
    .filter(col("EffectiveDate") < col("EndDate"))

In [0]:
df_dim_broker = spark.table(f"{catalog}.{gold_schema}.dim_broker") \
    .select(
        col("brokerid").cast("bigint").alias("broker_natural_id"),
        col("sk_brokerid"),
    )

df_with_broker = df_final_scd2.alias("a") \
    .join(
        df_dim_broker.alias("b"),
        (col("a.CA_B_ID") == col("b.broker_natural_id")),
        "left"
    ) \
    .select("a.*", col("b.sk_brokerid").alias("SK_BrokerID"))

In [0]:
df_dim_account = df_with_broker.select(
    concat(date_format(col("EffectiveDate"), "yyyyMMdd"), col("AccountID").cast("string")).cast("bigint").alias("SK_AccountID"),
    col("AccountID"),
    col("SK_BrokerID"),
    col("SK_CustomerID"),
    col("AccountDesc"),
    col("TaxStatus"),
    col("Status"),
    col("IsCurrent"),
    col("EffectiveDate").alias("valid_from"),
    col("EndDate").alias("valid_to"),
    col("EffectiveDate"),
    col("EndDate"),
    col("version_number").cast("bigint").alias("version_number"),
    col("record_hash"),
    current_timestamp().alias("system_valid_from"),
    lit("9999-12-31T23:59:59").cast("timestamp").alias("system_valid_to"),
    col("_batch"),
    current_timestamp().alias("_load_ts"),
)

print(f"dim_account count: {df_dim_account.count()}")
target_table = f"{catalog}.{gold_schema}.dim_account"

In [0]:
if spark.catalog.tableExists(target_table):
    delta_target = DeltaTable.forName(spark, target_table)
    delta_target.alias("tgt").merge(df_dim_account.alias("src"), "tgt.AccountID = src.AccountID AND tgt.EffectiveDate = src.EffectiveDate") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    df_dim_account.write.saveAsTable(target_table)
    print("table created for gold.dim_account")

   

In [0]:
dim_account_count = spark.table(target_table).count()
print(f"Total rows: {dim_account_count}")

log_audit("gold.dim_account", "merge", dim_account_count)
log_checkpoint("gold_dim_account", "completed", dim_account_count)

# gold.fact_cash_balances

In [0]:
log_checkpoint("gold_fact_cash_balances", "in_progress")

df_fact_ct = spark.table(f"{catalog}.{gold_schema}.fact_cash_transactions")

df_daily = df_fact_ct.withColumn("DateValue", col("TransactionDatetime").cast("date")) \
    .groupBy("SK_AccountID", "DateValue") \
    .agg(sum("Amount").alias("DailyAmount"))

In [0]:
balance_window = Window.partitionBy("SK_AccountID").orderBy("DateValue") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_balances = df_daily.withColumn("Cash", sum("DailyAmount").over(balance_window))

df_balances = df_balances.withColumn("SK_DateID", date_format("DateValue", "yyyyMMdd").cast("bigint"))

df_dim_acc_current = spark.table(f"{catalog}.{gold_schema}.dim_account") \
    .filter(col("IsCurrent") == True) \
    .select(
        col("AccountID").alias("dim_account_id"),
        col("SK_CustomerID"),
    )

df_fact_balances = df_balances.alias("b") \
    .join(df_dim_acc_current.alias("d"),
        col("b.SK_AccountID") == col("d.dim_account_id"), "left") \
    .select(
        col("b.SK_AccountID").alias("AccountID"),
        col("b.DateValue").alias("DateValue"),
        col("d.SK_CustomerID").alias("SK_CustomerID"),
        col("b.SK_AccountID").alias("SK_AccountID"),
        col("b.SK_DateID").alias("SK_DateID"),
        col("b.Cash").cast("decimal(15,2)").alias("Cash"),
        lit(batch_id).alias("_batch"),
        current_timestamp().alias("_load_ts"),
        lit(run_id).alias("_run_id"),
    )

print(f"fact_cash_balances count: {df_fact_balances.count()}")

In [0]:
target_table = f"{catalog}.{gold_schema}.fact_cash_balances"

# Deduplicate on merge key to avoid multiple source rows matching same target
df_fact_balances_deduped = df_fact_balances.dropDuplicates(["SK_AccountID", "SK_DateID"])

if spark.catalog.tableExists(target_table):
    delta_target = DeltaTable.forName(spark, target_table)
    delta_target.alias("tgt").merge(
        df_fact_balances_deduped.alias("src"),
        "tgt.SK_AccountID = src.SK_AccountID AND tgt.SK_DateID = src.SK_DateID"
    ) \
    .whenMatchedUpdate(
        set={
            "Cash": "src.Cash",
            "SK_CustomerID": "src.SK_CustomerID",
            "AccountID": "src.AccountID",
            "DateValue": "src.DateValue",
            "_batch": "src._batch",
            "_load_ts": "src._load_ts",
            "_run_id": "src._run_id",
        }        
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

    print("gold.fact_cash_balances done")
else:
    df_fact_balances_deduped.write.saveAsTable(target_table)
    print("gold.fact_cash_balances done")

In [0]:
fact_bal_count = spark.table(target_table).count()
print(f"Total rows: {fact_bal_count}")
log_audit("gold.fact_cash_balances", "merge", fact_bal_count)
log_checkpoint("gold.fact_cash_balances", "completed", fact_bal_count)

In [0]:
display(spark.table(f"{catalog}.{gold_schema}.fact_cash_balances"))